# qorder: experimentos no Colab

Rode as células na ordem. Os resultados ficam no Google Drive, então uma sessão interrompida continua de onde parou: o runner pula os modelos já treinados.

**Ambiente de execução:** a CPU padrão basta. Para 8 qubits, a GPU não acelera o PennyLane.

## 1. Código e dependências

In [ ]:
# Opção A: repositório no GitHub (troque pela URL do grupo)
# !git clone https://github.com/<usuario>/qorder.git
# %cd qorder

# Opção B: enviar o qorder.zip pelo painel de arquivos do Colab e descompactar
import os
if not os.path.exists('qorder'):
    !unzip -q qorder.zip
%cd /content/qorder

In [ ]:
!pip install -q -r requirements.txt

## 2. Testes

Os testes de `test_equivalence.py` conferem se o circuito em PennyLane, o mesmo circuito em Qiskit e o simulador NumPy de referência dão os mesmos valores esperados. **Se algum teste falhar, não rode os experimentos:** mandem a saída para revisão.

In [ ]:
!python -m pytest -q

## 3. Pasta de resultados no Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RESULTS = '/content/drive/MyDrive/qorder_results'
os.makedirs(RESULTS, exist_ok=True)

## 4. Teste rápido do pipeline (~2 min)

In [ ]:
!python -m qorder.run configs/exp2_breast_cancer_shallow.yaml --quick --results $RESULTS
!python -m qorder.analysis $RESULTS/exp2_breast_cancer_shallow_quick

## 5. Experimentos completos

Cada célula pode levar de dezenas de minutos a algumas horas. Se a sessão cair, rode a mesma célula de novo.

In [ ]:
# H1, dados reais (circuito raso de 8 qubits)
!python -m qorder.run configs/exp2_breast_cancer_shallow.yaml --results $RESULTS
!python -m qorder.run configs/exp2_mnist_shallow.yaml --results $RESULTS
!python -m qorder.run configs/exp2_fashion_shallow.yaml --results $RESULTS

In [ ]:
# H1, dataset adversarial com o modelo de Salmenperä et al. (o mais demorado)
!python -m qorder.run configs/exp1_adversarial_drc.yaml --results $RESULTS

In [ ]:
# H2, profundidade e custo global
!python -m qorder.run configs/exp3_depth_cost.yaml --results $RESULTS

In [ ]:
# H3, variância do gradiente
!python -m qorder.run configs/exp4_gradients.yaml --results $RESULTS

## 6. Análise

In [ ]:
import pandas as pd
from IPython.display import Image, display
for exp in ['exp2_breast_cancer_shallow', 'exp2_mnist_shallow', 'exp2_fashion_shallow',
            'exp1_adversarial_drc', 'exp3_depth_cost', 'exp4_gradients']:
    d = f'{RESULTS}/{exp}'
    if os.path.exists(d):
        print('=' * 20, exp)
        !python -m qorder.analysis $d
        for img in ('orderings.png', 'gradients.png'):
            if os.path.exists(f'{d}/{img}'):
                display(Image(f'{d}/{img}'))

## 7. Validação: shots, ruído simulado e hardware

Rode primeiro `ideal` e `fake:auto`, que são gratuitos e não usam a cota da IBM. Eles mostram quanto o ruído e o número de shots devem reduzir a acurácia antes de gastar tempo de QPU.

In [ ]:
EXP = f'{RESULTS}/exp2_breast_cancer_shallow'
!python scripts/validate_hardware.py $EXP --mode ideal    --orderings heuristic default --layers 2 --shots 1000
!python scripts/validate_hardware.py $EXP --mode fake:auto --orderings heuristic default --layers 2 --shots 1000

### Hardware IBM (gasta a cota mensal)

1. Guarde o token da IBM Quantum em **Secrets** do Colab (ícone de chave) com o nome `IBM_QUANTUM_TOKEN`. Se precisarem, guardem também a instância (CRN) em `IBM_QUANTUM_INSTANCE`.
2. Rode antes com `--dry-run` para ver o tempo estimado.
3. Mantenham `--budget`: o script se recusa a enviar o job se a estimativa passar desse limite.

In [ ]:
from google.colab import userdata
os.environ['IBM_QUANTUM_TOKEN'] = userdata.get('IBM_QUANTUM_TOKEN')
try:
    os.environ['IBM_QUANTUM_INSTANCE'] = userdata.get('IBM_QUANTUM_INSTANCE')
except Exception:
    pass
ARGS = '--orderings heuristic default --layers 2 --seeds 0 --shots 1000 --max-samples 200'
!python scripts/validate_hardware.py $EXP --mode ibm:least_busy $ARGS --dry-run

In [ ]:
# Envia de verdade: retire o comentário quando o grupo decidir
# !python scripts/validate_hardware.py $EXP --mode ibm:least_busy $ARGS --budget 150